In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad('/data2/home/vcivale/scfm-layer-analysis-refactored/data/embeddings/inner_ear_development_tahoe_embeddings.h5ad')

In [ ]:
"""
Compare UMAP visualizations between best and last layer with multiple annotations.
"""
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np

# Assuming adata is already loaded and has layer embeddings

# Define layers
best_layer = 13
last_layer = 23

print(f"Computing UMAPs for layer {best_layer} (best) and layer {last_layer} (last)...")

# ===== BEST LAYER =====
print(f"\nProcessing layer {best_layer}...")
sc.pp.neighbors(adata, use_rep=f'X_layer_{best_layer}', n_neighbors=15)
sc.tl.umap(adata)
adata.obsm[f'X_umap_layer{best_layer}'] = adata.obsm['X_umap'].copy()

# Compute DPT for best layer (optional, if not already computed)
if 'dpt_pseudotime' not in adata.obs.columns:
    print("  Computing diffusion pseudotime...")
    # Find root cell (earliest timepoint)
    if 'week' in adata.obs.columns:
        min_time = adata.obs['week'].min()
        root_idx = np.where(adata.obs['week'] == min_time)[0][0]
        adata.uns['iroot'] = root_idx
        sc.tl.dpt(adata)
    else:
        print("  Warning: 'week' column not found, skipping DPT")

# ===== LAST LAYER =====
print(f"\nProcessing layer {last_layer}...")
sc.pp.neighbors(adata, use_rep=f'X_layer_{last_layer}', n_neighbors=15)
sc.tl.umap(adata)
adata.obsm[f'X_umap_layer{last_layer}'] = adata.obsm['X_umap'].copy()

# ===== PLOTTING =====
print("\nGenerating comparison plots...")

# Determine available metadata
metadata_to_plot = []
if 'week' in adata.obs.columns:
    metadata_to_plot.append('week')
if 'timepoint' in adata.obs.columns:
    metadata_to_plot.append('timepoint')
if 'developmental_stage' in adata.obs.columns:
    metadata_to_plot.append('developmental_stage')
if 'dpt_pseudotime' in adata.obs.columns:
    metadata_to_plot.append('dpt_pseudotime')

# If no temporal metadata, use what's available
if not metadata_to_plot:
    metadata_to_plot = ['sample', 'batch']  # fallback
    print("Warning: No temporal metadata found, using sample/batch")

n_cols = len(metadata_to_plot)

# Create figure
fig, axes = plt.subplots(2, n_cols, figsize=(6*n_cols, 12))

# Ensure axes is 2D
if n_cols == 1:
    axes = axes.reshape(-1, 1)

print(f"Plotting {n_cols} metadata columns...")

# Row 1: Best layer
adata.obsm['X_umap'] = adata.obsm[f'X_umap_layer{best_layer}']
for i, meta in enumerate(metadata_to_plot):
    if meta == 'dpt_pseudotime':
        sc.pl.umap(adata, color=meta, ax=axes[0, i], show=False, 
                  title=f'Layer {best_layer} - Pseudotime', 
                  cmap='viridis', colorbar_loc='right')
    else:
        sc.pl.umap(adata, color=meta, ax=axes[0, i], show=False,
                  title=f'Layer {best_layer} - {meta}', 
                  legend_loc='on data' if 'stage' in meta else 'right margin')

# Row 2: Last layer
adata.obsm['X_umap'] = adata.obsm[f'X_umap_layer{last_layer}']
for i, meta in enumerate(metadata_to_plot):
    if meta == 'dpt_pseudotime':
        sc.pl.umap(adata, color=meta, ax=axes[1, i], show=False,
                  title=f'Layer {last_layer} - Pseudotime',
                  cmap='viridis', colorbar_loc='right')
    else:
        sc.pl.umap(adata, color=meta, ax=axes[1, i], show=False,
                  title=f'Layer {last_layer} - {meta}',
                  legend_loc='on data' if 'stage' in meta else 'right margin')

plt.tight_layout()
plt.savefig('layer_comparison_umap.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Saved: layer_comparison_umap.png")
plt.show()

# ===== ADDITIONAL: Side-by-side single metadata =====
# Create focused comparison for main temporal variable
if 'week' in adata.obs.columns:
    main_temporal = 'week'
elif 'timepoint' in adata.obs.columns:
    main_temporal = 'timepoint'
else:
    main_temporal = metadata_to_plot[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Best layer
adata.obsm['X_umap'] = adata.obsm[f'X_umap_layer{best_layer}']
sc.pl.umap(adata, color=main_temporal, ax=axes[0], show=False,
          title=f'Layer {best_layer} (Best) - {main_temporal}',
          frameon=True, legend_loc='right margin')

# Last layer
adata.obsm['X_umap'] = adata.obsm[f'X_umap_layer{last_layer}']
sc.pl.umap(adata, color=main_temporal, ax=axes[1], show=False,
          title=f'Layer {last_layer} (Last) - {main_temporal}',
          frameon=True, legend_loc='right margin')

plt.tight_layout()
plt.savefig('layer_comparison_simple.png', dpi=300, bbox_inches='tight')
print(f"✓ Saved: layer_comparison_simple.png")
plt.show()

print("\n" + "="*60)
print("Comparison complete!")
print(f"Best layer: {best_layer}")
print(f"Last layer: {last_layer}")
print(f"Metadata plotted: {', '.join(metadata_to_plot)}")
print("="*60)

In [ ]:
import scanpy as sc
adata_pert = sc.read_h5ad("data/embeddings/D4_Rest_undersampled_tahoe.h5ad", backed='r')
adata_pert

In [ ]:
print("Guide types:")
print(adata_pert.obs['guide_type'].value_counts())

print("\nGuide groups:")
print(adata_pert.obs['guide_group'].value_counts())

print("\nSample perturbed_gene_name:")
print(adata_pert.obs['perturbed_gene_name'].value_counts().head(20))

In [ ]:
import numpy as np

for key in adata_pert.obsm.keys():
    arr = adata_pert.obsm[key]
    if np.isnan(arr).any():
        print(f"NaN trovati in obsm['{key}']")
    else:
        print(f"Nessun NaN in obsm['{key}']")

In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/data/human_cd34_bm_rep1.h5ad')

In [ ]:
import mygene

mg = mygene.MyGeneInfo()
gene_symbols = adata.var_names.tolist()

batch_size = 500
results = []

for i in range(0, len(gene_symbols), batch_size):
    batch = gene_symbols[i:i+batch_size]
    res = mg.querymany(batch, scopes='symbol', fields='ensembl.gene', species='human')
    results.extend(res)

# Ora processa results come prima
symbol_to_ensembl = {}
for res in results:
    symbol = res['query']
    if 'ensembl' in res:
        ensembl_data = res['ensembl']
        if isinstance(ensembl_data, list):
            ensembl_id = ensembl_data[0]['gene']
        else:
            ensembl_id = ensembl_data['gene']
        symbol_to_ensembl[symbol] = ensembl_id
    else:
        symbol_to_ensembl[symbol] = None

In [ ]:
import pandas as pd

ensembl_ids = [symbol_to_ensembl.get(g, None) for g in gene_symbols]

# Sostituisco None con stringa vuota
ensembl_ids_str = [str(eid) if eid is not None else '' for eid in ensembl_ids]

adata.var_names = pd.Series(ensembl_ids_str, index=adata.var_names)

# Ora salva senza errori
adata.write_h5ad('data/raw/human_cd34_bm_rep1.h5ad')


In [ ]:
adata = sc.read_h5ad('data/embeddings/human_cd34_bm_rep1_tahoe_1b_embeddings.h5ad')

In [ ]:
adata.X = adata.raw.X

In [ ]:
adata.write_h5ad('data/raw/human_cd34_bm_rep1_tahoe_1b_embeddings.h5ad')

In [ ]:
import numpy as np

# Rendi univoci i nomi dei geni
adata.var_names_make_unique()

# Rimuovi le celle con tutti zeri
adata = adata[adata.X.sum(axis=1) > 0].copy()

In [ ]:
adata.write_h5ad('data/raw/human_cd34_bm_rep1_tahoe_1b_embeddings.h5ad')

In [ ]:
import numpy as np

# Rimuovi geni con NaN
X = adata.X
if hasattr(X, "toarray"):  # sparse matrix
	X = X.toarray()
X = np.asarray(X, dtype=np.float64)
mask = ~np.isnan(X).any(axis=0)
adata = adata[:, mask].copy()

In [ ]:
from datasets import load_dataset
import random

# Carica il dataset in streaming (puoi cambiare split a seconda del bisogno)
ds = load_dataset("Xaira-Therapeutics/X-Atlas-Orion", streaming=True)

# Set guida di interesse (puoi modificarlo o metterlo a None per tutte)
guides_of_interest = {"guideA", "guideB", "guideC"}  # es. metti guide reali, o None

# Funzione per filtrare cellule con pass_guide_filter True e guida di interesse
def filter_cell(cell):
    if not cell.get('pass_guide_filter', False):
        return False
    if guides_of_interest is None:
        return True
    return cell.get('guide_target') in guides_of_interest

# Filtra le cellule
filtered_cells = (cell for cell in ds if filter_cell(cell))

# Prendi un subsample casuale di dimensione max_sample_size (es. 1000)
max_sample_size = 1000
subsample = []
count_filtered = 0

for cell in filtered_cells:
    count_filtered += 1
    if len(subsample) < max_sample_size:
        subsample.append(cell)
    else:
        # Reservoir sampling per mantenere campione casuale tra tutti
        r = random.randint(0, count_filtered - 1)
        if r < max_sample_size:
            subsample[r] = cell

print(f"Totale cellule filtrate: {count_filtered}")
print(f"Dimensione subsample estratto: {len(subsample)}")


In [ ]:
adata_or = sc.read_h5ad('data/raw/human_cd34_bm_rep1.h5ad')

In [ ]:
adata = sc.read_h5ad('data/embeddings/human_cd34_bm_rep1_tahoe_1b_embeddings.h5ad')

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "Xaira-Therapeutics/X-Atlas-Orion",
    streaming=True
)


In [ ]:
ds

In [ ]:
hct = load_dataset(
    "Xaira-Therapeutics/X-Atlas-Orion",
    split="HCT116",
    streaming=True
)

from collections import Counter

gene_counter = Counter()
guide_counter = Counter()

for i, cell in enumerate(hct):
    gene_counter[cell["gene_target"]] += 1
    guide_counter[cell["guide_target"]] += 1
    if i > 200_000:
        break


In [ ]:
len(gene_counter), len(guide_counter)


In [ ]:
from collections import Counter

gene_counts = Counter()

for cell in hct:  # streaming
    if cell["pass_guide_filter"]:
        gene_counts[cell["gene_target"]] += 1


In [ ]:
from datasets import load_dataset
from collections import Counter

hct = load_dataset(
    "Xaira-Therapeutics/X-Atlas-Orion",
    split="HCT116",
    streaming=True
)

gene_counts = Counter()

for i, cell in enumerate(hct):
    if cell["pass_guide_filter"]:
        gene_counts[cell["gene_target"]] += 1
    if i == 3_000_000:
        break



In [ ]:
import numpy as np

counts = np.array(list(gene_counts.values()))

p25, p50, p75 = np.percentile(counts, [25, 50, 75])
p25, p50, p75


In [ ]:
low, mid, high = [], [], []

for gene, n in gene_counts.items():
    if n <= p25:
        low.append(gene)
    elif n <= p75:
        mid.append(gene)
    else:
        high.append(gene)

len(low), len(mid), len(high)


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from datasets import load_dataset
import anndata
from scipy.sparse import lil_matrix
import random

random.seed(42)
np.random.seed(42)

# --- PARAMETRI ---
N_LOW = 500
N_MID = 1000
N_HIGH = 500

MAX_CELLS_LOW = None   # tutte
MAX_CELLS_MID = 15
MAX_CELLS_HIGH = 20

# --- PASSO 1: Conta cellule per perturbazione (gene_target) ---

print("Counting cells per gene_target...")

hct = load_dataset("Xaira-Therapeutics/X-Atlas-Orion", split="HCT116", streaming=True)

gene_counts = Counter()
for i, cell in enumerate(hct):
    if cell["pass_guide_filter"]:
        gene_counts[cell["gene_target"]] += 1
    if i > 2_000_000:  # early stop per velocità
        break

counts = np.array(list(gene_counts.values()))
p25, p50, p75 = np.percentile(counts, [25, 50, 75])
print(f"Percentili (p25, p50, p75): {p25}, {p50}, {p75}")

# --- PASSO 2: Definisci strati ---

low_genes = [g for g, c in gene_counts.items() if c <= p25]
mid_genes = [g for g, c in gene_counts.items() if p25 < c <= p75]
high_genes = [g for g, c in gene_counts.items() if c > p75]

print(f"Low abundance genes: {len(low_genes)}")
print(f"Mid abundance genes: {len(mid_genes)}")
print(f"High abundance genes: {len(high_genes)}")

# --- PASSO 3: Seleziona random stratificato perturbazioni ---

selected_low = random.sample(low_genes, min(N_LOW, len(low_genes)))
selected_mid = random.sample(mid_genes, min(N_MID, len(mid_genes)))
selected_high = random.sample(high_genes, min(N_HIGH, len(high_genes)))

selected_genes = set(selected_low + selected_mid + selected_high)
print(f"Selected genes total: {len(selected_genes)}")

# --- PASSO 4: Setup max cellule per gene ---

max_cells_per_gene = {}
for g in selected_low:
    max_cells_per_gene[g] = MAX_CELLS_LOW
for g in selected_mid:
    max_cells_per_gene[g] = MAX_CELLS_MID
for g in selected_high:
    max_cells_per_gene[g] = MAX_CELLS_HIGH

# --- PASSO 5: Estrazione dati cellula per cellula ---

print("Extracting cells for selected perturbations...")

ds = load_dataset("Xaira-Therapeutics/X-Atlas-Orion", split="HCT116", streaming=True)

cell_counts = Counter()
cell_data = []
gene_token_set = set()

for cell in ds:
    if not cell["pass_guide_filter"]:
        continue
    gene = cell["gene_target"]
    if gene not in selected_genes:
        continue

    if max_cells_per_gene[gene] is not None and cell_counts[gene] >= max_cells_per_gene[gene]:
        continue

    cell_data.append(cell)
    cell_counts[gene] += 1

print(f"Extracted {len(cell_data)} cells in total.")

# --- PASSO 6: Costruzione matrice espressione sparsa ---

for cell in cell_data:
    gene_token_set.update(cell["gene_token_id"])

gene_token_list = sorted(gene_token_set)
gene_token_to_idx = {g: i for i, g in enumerate(gene_token_list)}

n_cells = len(cell_data)
n_genes = len(gene_token_list)

print(f"Building sparse matrix: {n_cells} cells × {n_genes} genes...")

X = lil_matrix((n_cells, n_genes), dtype=np.float32)

for i, cell in enumerate(cell_data):
    gene_ids = cell["gene_token_id"]
    exprs = cell["gene_expression"]
    for g_id, expr in zip(gene_ids, exprs):
        j = gene_token_to_idx[g_id]
        X[i, j] = expr

# --- PASSO 7: Prepara metadata cellule ---

obs = pd.DataFrame({
    "cell_barcode": [c["cell_barcode"] for c in cell_data],
    "sample": [c["sample"] for c in cell_data],
    "guide_target": [c["guide_target"] for c in cell_data],
    "gene_target": [c["gene_target"] for c in cell_data],
    "n_genes_by_counts": [c["n_genes_by_counts"] for c in cell_data],
    "total_counts": [c["total_counts"] for c in cell_data],
    "total_counts_mt": [c["total_counts_mt"] for c in cell_data],
    "pct_counts_mt": [c["pct_counts_mt"] for c in cell_data],
    "pass_guide_filter": [c["pass_guide_filter"] for c in cell_data],
})

# --- PASSO 8: Carica metadata geni ---

gene_metadata = load_dataset("Xaira-Therapeutics/X-Atlas-Orion", "gene_metadata")
gene_meta_df = pd.DataFrame(gene_metadata)
gene_meta_sub = gene_meta_df[gene_meta_df["gene_token_id"].isin(gene_token_list)].copy()
gene_meta_sub = gene_meta_sub.set_index("gene_token_id").loc[gene_token_list]

# --- PASSO 9: Costruisci AnnData e salva ---

adata = anndata.AnnData(X=X.tocsr(), obs=obs, var=gene_meta_sub)

output_path = "XAtlasOrion_HCT116_subset.h5ad"
print(f"Saving AnnData to {output_path} ...")
adata.write(output_path)
print("Done.")


In [ ]:
DEBUG = True
MAX_DEBUG_CELLS = 500


In [ ]:
for i, cell in enumerate(ds):
    if not cell["pass_guide_filter"]:
        continue
    gene = cell["gene_target"]
    if gene not in selected_genes:
        continue

    if max_cells_per_gene[gene] is not None and cell_counts[gene] >= max_cells_per_gene[gene]:
        continue

    cell_data.append(cell)
    cell_counts[gene] += 1

    if DEBUG and len(cell_data) >= MAX_DEBUG_CELLS:
        break


In [ ]:
print("QC CHECKS")
print("Cells:", X.shape[0])
print("Genes:", X.shape[1])
print("obs rows:", obs.shape[0])
print("var rows:", gene_meta_sub.shape[0])

assert X.shape[0] == obs.shape[0]
assert X.shape[1] == gene_meta_sub.shape[0]
assert obs["gene_target"].nunique() <= 2000


In [ ]:
gene_metadata = load_dataset(
    "Xaira-Therapeutics/X-Atlas-Orion",
    "gene_metadata"
)

gene_meta_df = gene_metadata["train"].to_pandas()


In [ ]:
gene_meta_df.head()

In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/data/GSE276896_adata_meta.h5ad')

In [ ]:
import mygene
# Prendi i nomi dei geni attuali
genes = adata.var_names.tolist()

# Inizializza mygene query
mg = mygene.MyGeneInfo()

# Query batch per simboli genici (ad esempio specie umano)
query_res = mg.querymany(genes, scopes='symbol', fields='ensembl.gene', species='human')

# Crea dizionario mapping simbolo -> ensembl_id (prendi il primo Ensembl ID se multipli)
symbol2ensembl = {}
for entry in query_res:
    if 'notfound' in entry and entry['notfound']:
        continue
    ensembl_id = None
    if 'ensembl' in entry:
        if isinstance(entry['ensembl'], list):
            ensembl_id = entry['ensembl'][0]['gene']
        else:
            ensembl_id = entry['ensembl']['gene']
    if ensembl_id:
        symbol2ensembl[entry['query']] = ensembl_id

# Mappa i nomi originali in Ensembl
new_var_names = [symbol2ensembl.get(gene, gene) for gene in genes]

# Sostituisci var_names in AnnData
adata.var_names = new_var_names


In [ ]:
import scanpy as sc

# Percorso alla cartella contenente i file barcodes.tsv.gz, features.tsv.gz e matrix.mtx.gz
data_dir = "/data/GSE297393/GSM8989955_UT"  # esempio, cambia secondo la directory giusta

# Carica la matrice 10x
adata = sc.read_10x_mtx(data_dir,  # cartella contenente i 3 file
                        var_names='gene_symbols',  # o 'gene_ids' se vuoi usare gli Ensembl ID
                        cache=True)

# Controlla info
print(adata)

In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/home/oem/scfm-layer-analysis-refactored/data/embeddings/GSE276896_adata_meta_tahoe_1b_embeddings.h5ad')
adata

In [ ]:
adata.obs['Timepoint'] = pd.Categorical(
    adata.obs['Timepoint'],
    categories=['Day3', 'Day10', 'Day17'],
    ordered=True
)

In [ ]:
adata.obs['Timepoint']

In [ ]:
adata.obs.head()


In [ ]:
adata.obs['transferred_labels'].value_counts()

In [ ]:
adata.write('/home/oem/scfm-layer-analysis-refactored/data/embeddings/GSE276896_adata_meta_tahoe_1b_embeddings.h5ad')

In [ ]:
import scanpy as sc
adata_raw = sc.read_h5ad('/home/oem/scfm-layer-analysis-refactored/GSE276896_adata_meta.h5ad')

In [ ]:
adata_raw.var['feature_name'] = adata_raw.var_names

In [ ]:
adata_raw.write('/home/oem/scfm-layer-analysis-refactored/GSE276896_adata_meta.h5ad')

In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/data2/home/vcivale/scfm-layer-analysis-refactored/data/embeddings/liver_dataset_scfoundation_embeddings.h5ad')

In [ ]:
adata

In [ ]:
adata

In [ ]:
adata2 = sc.read_h5ad('data/raw/liver_dataset.h5ad')

In [ ]:
adata2.obsm = adata.obsm

In [ ]:
adata2.write_h5ad('/data2/home/vcivale/scfm-layer-analysis-refactored/data/embeddings/liver_dataset_scfoundation_embeddings.h5ad')

In [ ]:
adata2

In [ ]:
import matplotlib.pyplot as plt

# Escludi baseline e righe senza layer_idx
df_plot = df[df["layer_idx"].notna()].copy()

# Ordina per indice numerico
df_plot = df_plot.sort_values("layer_idx").reset_index(drop=True)

# Profondità percentuale
max_layer = df_plot["layer_idx"].max()
df_plot["depth_percent"] = df_plot["layer_idx"] / max_layer * 100


plt.figure(figsize=(8, 5))
plt.plot(
    df_plot["depth_percent"],
    df_plot["F1_macro"],
    marker="o",
    label="F1_macro"
)
plt.xlabel("Profondità della rete (%)")
plt.ylabel("F1 score")
plt.title("Andamento Recall score con la profondità della rete")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Caricamento CSV
df = pd.read_csv("/data2/home/vcivale/scfm-layer-analysis-refactored/data/pseudotime_results/GSE276896_adata_meta_tahoe_1b_embeddings_results.csv")

# Estrazione indice numerico del layer
def extract_layer_idx(layer_name):
    match = re.search(r"X_layer_(\d+)", layer_name)
    return int(match.group(1)) if match else None

df["layer_idx"] = df["Layer"].apply(extract_layer_idx)



# Ordina correttamente per indice numerico
df = df.sort_values("layer_idx").reset_index(drop=True)

In [ ]:
df.to_csv("/data2/home/vcivale/scfm-layer-analysis-refactored/data/pseudotime_results/GSE276896_adata_meta_tahoe_1b_embeddings_results.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

def plot_models_from_csvs(
    csv_paths,
    model_names,
    metric,
    colors,
    markers=None,
    figsize=(7, 4.5),
    title=None
):
    """
    Confronto di più modelli a partire dai rispettivi CSV.
    """

    if markers is None:
        markers = ["o", "s", "^", "D"]

    plt.figure(figsize=figsize)

    for i, (csv_path, model_name) in enumerate(zip(csv_paths, model_names)):
        df = pd.read_csv(csv_path)

        # Estrazione indice numerico del layer
        def extract_layer_idx(layer_name):
            match = re.search(r"X_layer_(\d+)", layer_name)
            return int(match.group(1)) if match else None

        df["layer_idx"] = df["Layer"].apply(extract_layer_idx)



        # Ordina correttamente per indice numerico
        df = df.sort_values("layer_idx").reset_index(drop=True)

        # Usa solo layer validi
        df = df[df["layer_idx"].notna()].copy()
        df = df.sort_values("layer_idx")

        # Profondità percentuale
        max_layer = df["layer_idx"].max()
        depth_percent = df["layer_idx"] / max_layer * 100

        plt.plot(
            depth_percent,
            df[metric],
            label=model_name,
            color=colors[i],
            marker=markers[i % len(markers)],
            linewidth=2.2,
            markersize=5
        )

    plt.xlabel("Layer Depth Percentage")
    plt.ylabel(metric.replace("_", " "))
    
    if title:
        plt.title(title)

    plt.legend(frameon=False)
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


In [ ]:
csvs = [
    "/data2/home/vcivale/scfm-layer-analysis-refactored/data/pseudotime_results/GSE276896_adata_meta_scfoundation_embeddings_results.csv",
    "/data2/home/vcivale/scfm-layer-analysis-refactored/data/pseudotime_results/GSE276896_adata_meta_tahoe_1b_embeddings_results.csv"
]


models = ["scFoundation", "Tahoe-1B"]
colors = ["#0072B2", "#D55E00"]  # palette paper-safe

plot_models_from_csvs(
    csv_paths=csvs,
    model_names=models,
    metric="Pseudotime_Corr",
    colors=colors,
    title="Embedding Quality vs Network Depth (Dataset A)"
)


In [2]:
import scanpy as sc
adata = sc.read_h5ad("/data2/home/vcivale/scfm-layer-analysis-refactored/data/embeddings/D4_Rest_undersampled_tahoe.h5ad", backed='r')

In [3]:
adata

AnnData object with n_obs × n_vars = 700000 × 18130 backed at '/data2/home/vcivale/scfm-layer-analysis-refactored/data/embeddings/D4_Rest_undersampled_tahoe.h5ad'
    obs: 'lane_id', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'top_guide_UMI_counts', 'guide_id', 'perturbed_gene_name', 'perturbed_gene_id', 'guide_type', 'PuroR', 'guide_group', 'low_quality'
    obsm: 'X_layer_0', 'X_layer_1', 'X_layer_10', 'X_layer_11', 'X_layer_12', 'X_layer_13', 'X_layer_14', 'X_layer_15', 'X_layer_16', 'X_layer_17', 'X_layer_18', 'X_layer_19', 'X_layer_2', 'X_layer_20', 'X_layer_21', 'X_layer_22', 'X_layer_23', 'X_layer_3', 'X_layer_4', 'X_layer_5', 'X_layer_6', 'X_layer_7', 'X_layer_8', 'X_layer_9'

In [4]:
data = adata.obs['perturbed_gene_name'].value_counts()
for gene, count in data.items():
    print(f"{gene}: {count}")

NTC: 19785
LRP2BP: 560
ELOF1: 338
NFKBIL1: 327
PARP14: 273
EDC3: 263
CYP20A1: 236
SETDB1: 234
TANC2: 231
RASAL3: 221
WDR54: 218
ZMAT1: 214
PIGB: 206
RFX5: 201
NR1H2: 197
MAPK8: 181
ACER2: 177
TENT2: 177
MAP3K7CL: 168
SLC38A7: 165
LILRA2: 164
EPHA4: 163
POLK: 160
ABHD14B: 158
BRI3BP: 157
XPNPEP1: 157
TRAF3: 156
TADA2B: 156
NANOG: 153
HSDL2: 153
CASP3: 152
PTK2B: 150
ZNF613: 148
MYB: 146
FBXO44: 145
CISH: 145
IFIT5: 144
SCOC: 144
DEFA6: 142
RB1: 142
SFT2D2: 140
PPDPF: 140
ORM2: 139
RTKN2: 139
RAB27B: 138
TCEA1: 137
DDX58: 136
FYTTD1: 136
RAB8B: 135
TRANK1: 135
TWF2: 135
HLCS: 134
ATAD5: 134
CCDC15: 133
EIF1AX: 132
TMEM185A: 131
APBB3: 131
SUMO2: 131
DLG4: 130
LRRC37A3: 129
LTF: 129
UBL7: 129
MTERF2: 129
MTM1: 128
ECHS1: 127
ATXN7L3: 127
ALS2CL: 127
PAQR3: 126
PABPC1L: 125
TMPRSS13: 125
FAM78A: 124
SLC30A1: 124
EMX2: 124
NEK6: 123
PVRIG: 123
RCAN2: 123
SCN8A: 123
HIST1H2BI: 122
TESK2: 122
HIST1H4I: 122
CBLB: 122
FHL1: 121
ZNF426: 121
MLN: 121
IL36RN: 121
DCAKD: 121
ZNF784: 120
SP140: 120


In [5]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import anndata
import os

OUTPUT_PATH = 'D4_Rest_representative_subset.h5ad'
PERTURB_KEY = 'perturbed_gene_name'
CONTROL_LABEL = 'NTC' 
CELLS_PER_PERTURB = 500
CELLS_FOR_CONTROL = 3000
RANDOM_SEED = 42

# Numero massimo di perturbazioni da includere
MAX_PERTURBATIONS = 100

def create_representative_subset():
    print("Caricamento dei metadati (.obs)...")
    obs_df = adata.obs.copy()
    print(f"Trovate {adata.n_obs} cellule totali.")

    print(f"Stratificazione per colonna: '{PERTURB_KEY}'")
    grouped = obs_df.groupby(PERTURB_KEY)

    # Conta il numero di cellule per ogni perturbazione
    perturb_counts = grouped.size().sort_values(ascending=False)

    # Escludiamo il gruppo di controllo dalla selezione
    perturb_counts_no_control = perturb_counts.drop(CONTROL_LABEL, errors='ignore')

    # Selezioniamo le prime MAX_PERTURBATIONS perturbazioni più grandi
    top_perturbations = perturb_counts_no_control.head(MAX_PERTURBATIONS).index.tolist()
    print(f"Selezionate le prime {MAX_PERTURBATIONS} perturbazioni più numerose.")

    # Aggiungiamo sempre il controllo
    top_perturbations.append(CONTROL_LABEL)

    final_indices = []

    for name, group in tqdm(grouped, total=len(grouped), desc="Campionamento gruppi"):
        if name not in top_perturbations:
            continue  # salto perturbazioni non selezionate

        n_cells_in_group = len(group)
        if name == CONTROL_LABEL:
            n_to_sample = min(CELLS_FOR_CONTROL, n_cells_in_group)
            print(f"Gruppo di controllo '{name}': campionamento di {n_to_sample} cellule su {n_cells_in_group}.")
        else:
            n_to_sample = min(CELLS_PER_PERTURB, n_cells_in_group)

        sampled_indices = group.sample(n=n_to_sample, random_state=RANDOM_SEED).index
        final_indices.extend(sampled_indices)

    if pd.api.types.is_string_dtype(obs_df.index.dtype):
         final_indices_pos = obs_df.index.get_indexer(final_indices)
    else:
         final_indices_pos = final_indices

    print(f"\nIndici totali selezionati per il subset: {len(final_indices_pos)}")

    print("Caricamento del subset di dati in memoria (potrebbe richiedere tempo)...")
    adata_subset = adata[final_indices_pos, :].to_memory()

    print(f"Salvataggio del nuovo subset in: {OUTPUT_PATH}")
    adata_subset.write_h5ad(OUTPUT_PATH, compression="gzip")

    print("\n--- Completato! ---")
    print(f"Creato file '{OUTPUT_PATH}' con {adata_subset.n_obs} cellule.")

if __name__ == "__main__":
    create_representative_subset()


Caricamento dei metadati (.obs)...
Trovate 700000 cellule totali.
Stratificazione per colonna: 'perturbed_gene_name'
Selezionate le prime 100 perturbazioni più numerose.


/tmp/ipykernel_804679/3466612931.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = obs_df.groupby(PERTURB_KEY)
Campionamento gruppi:  57%|█████▋    | 6946/12107 [00:00<00:00, 26871.77it/s]

Gruppo di controllo 'NTC': campionamento di 3000 cellule su 19785.


Campionamento gruppi: 100%|██████████| 12107/12107 [00:00<00:00, 16685.53it/s]



Indici totali selezionati per il subset: 18264
Caricamento del subset di dati in memoria (potrebbe richiedere tempo)...
Salvataggio del nuovo subset in: D4_Rest_representative_subset.h5ad

--- Completato! ---
Creato file 'D4_Rest_representative_subset.h5ad' con 18264 cellule.
